In [2]:
import pandas as pd 
import numpy as np

In [3]:
import matplotlib.pyplot as plt
from pyts.image import GramianAngularField
import os
from sklearn.preprocessing import MinMaxScaler

In [ ]:
data=pd.read_csv('bitcoin_2022_2025.csv')

In [4]:
data.head()

,datetime_utc,open,high,low,close,volume,ema_9,ema_12,psar,rsi,obv,vwap,year,outlier,is_outlier,denoised,kalman,%K,%D
0,2022-01-01 00:00:00,46216.93,46391.49,46208.37,46321.34,185.67558,46344.729248,46352.040542,46486.778918,48.935750,-868452.529764,46307.066667,2022,1,False,47553.669892,46348.416101,48.777345,29.740508
1,2022-01-01 00:05:00,46321.34,46527.26,46280.00,46371.11,123.43577,46350.005399,46354.974304,46132.040000,51.678863,-868329.093994,46341.298104,2022,1,False,47553.847352,46350.574856,60.490360,43.713832
2,2022-01-01 00:10:00,46369.79,46394.00,46276.22,46332.51,77.54574,46346.506319,46351.518258,46139.944400,49.459733,-868406.639734,46339.883239,2022,1,False,47554.010477,46348.856438,50.723648,53.330451
3,2022-01-01 00:15:00,46332.52,46332.52,46236.27,46293.90,101.14315,46335.985055,46342.653910,46147.690712,47.273075,-868507.782884,46329.034946,2022,1,False,47554.159265,46343.628711,40.954405,50.722804
4,2022-01-01 00:20:00,46295.42,46421.27,46286.25,46395.53,135.32479,46347.894044,46350.788693,46155.282098,53.145136,-868372.458094,46337.428261,2022,1,False,47554.293713,46348.565817,66.669197,52.782417


In [5]:
gaf_data= data[['open', 'high', 'low', 'close', 'volume']].values

In [6]:
window_size = 20  # Matches GAF image_size=20
stride = 1        # Slide window by 1 timestep (adjust for overlap)

In [ ]:
# Normalize each OHLCV component to [-1, 1]
scaler = MinMaxScaler(feature_range=(-1, 1))
ohlcv_normalized = np.zeros_like(gaf_data)
for i in range(5):  # 5 columns: OHLCV
    ohlcv_normalized[:, i] = scaler.fit_transform(gaf_data[:, i].reshape(-1, 1)).flatten()

In [8]:

# Initialize GAF transformer
gaf = GramianAngularField(image_size=window_size, method='summation')

In [ ]:

# Create folders for each channel
os.makedirs('gaf_images/open', exist_ok=True)
os.makedirs('gaf_images/high', exist_ok=True)
os.makedirs('gaf_images/low', exist_ok=True)
os.makedirs('gaf_images/close', exist_ok=True)
os.makedirs('gaf_images/volume', exist_ok=True)

# Generate and save GAF images
for i in range(0, len(ohlcv_normalized) - window_size + 1, stride):
    window = ohlcv_normalized[i:i+window_size]
    timestamp = data.iloc[i]['datetime_utc'].replace(':', '-').replace(' ', '_')
    
    for j, col in enumerate(['open', 'high', 'low', 'close', 'volume']):
        # Generate GAF image for the current window and channel
        gaf_image = gaf.fit_transform(window[:, j].reshape(1, -1))[0]
        
        # Save as PNG
        plt.figure(figsize=(2, 2))
        plt.imshow(gaf_image, cmap='rainbow', origin='lower')
        plt.axis('off')
        plt.savefig(
            f'gaf_images/{col}/gaf_{timestamp}_window_{i:04d}.png',
            bbox_inches='tight',
            pad_inches=0,
            dpi=50
        )



KeyboardInterrupt: 

<Figure size 200x200 with 0 Axes>